# LSTM — S&P 500 Equity Straddles

LSTM's gating mechanism (forget, input, output gates) is designed for sequential
dependencies: it can learn that *how* IV changed over the past 60 days matters
for predicting delta-hedged returns, not just the current level. With 612
equity straddles producing the densest IV signal in the book, LSTM tests
whether volatility clustering and mean-reversion dynamics are better captured
by sequential memory than by flat feature interactions (TabM, GBM).

**Learning Objectives**:
- Quantify LSTM's advantage from sequential IV processing on options data
- Compare gated recurrence against patch-based attention (PatchTST)
- Assess whether LSTM's temporal processing adds value over Ridge (IC ~0.044)

**Book Reference**: Chapter 13

**Prerequisites**: [`06_linear`](06_linear.ipynb), [`07_gbm`](07_gbm.ipynb) (for comparison baselines)

In [1]:
"""LSTM — sp500_options deep learning."""

import warnings

import numpy as np
import polars as pl
import torch
import yaml

from case_studies.utils.analytics import load_best_ic_per_family
from case_studies.utils.deep_learning import (
    create_model,
    resolve_arch_name,
    run_dl_cv,
)
from utils.modeling import load_configs, load_modeling_dataset
from utils.paths import get_case_study_dir

warnings.filterwarnings("ignore")

In [2]:
CASE_STUDY_ID = "sp500_options"
MODEL = "lstm"
PRIMARY_LABEL = ""
MAX_SYMBOLS = 0
FORCE_RETRAIN = False  # Set True to retrain configs that already have complete hashes
PREDICTION_SPLIT = "validation"
N_EPOCHS = 100
LOOKBACK = 60
BATCH_SIZE = 2048
MC_DROPOUT = False
MAX_FOLDS = 0

In [3]:
CASE_DIR = get_case_study_dir(CASE_STUDY_ID)
setup = yaml.safe_load((CASE_DIR / "config" / "setup.yaml").read_text())

if not PRIMARY_LABEL:
    PRIMARY_LABEL = setup["labels"]["primary"]
    print(f"Label from setup.yaml: {PRIMARY_LABEL}")
else:
    print(f"Label override: {PRIMARY_LABEL}")

dl_config = setup.get("modeling", {}).get("dl", {})
DEVICE = dl_config.get("device", "gpu")

device_str = "cuda" if DEVICE == "gpu" and torch.cuda.is_available() else "cpu"
print(f"Case study: {CASE_STUDY_ID} | Model: {MODEL}")
print(f"Device: {device_str} | Epochs: {N_EPOCHS} | Lookback: {LOOKBACK}")

Label from setup.yaml: fwd_ret_dh_10d
Case study: sp500_options | Model: lstm
Device: cuda | Epochs: 100 | Lookback: 60


## 1. Load Data

In [4]:
mds = load_modeling_dataset(CASE_STUDY_ID, PRIMARY_LABEL, max_symbols=MAX_SYMBOLS)

dataset = mds.dataset
feature_names = mds.feature_names
label_col = mds.label_col
date_col = mds.date_col
entity_col = mds.entity_cols[0] if mds.entity_cols else "symbol"
splits = mds.splits
if MAX_FOLDS:
    splits = splits[:MAX_FOLDS]
n_features = len(feature_names)

print(f"Dataset: {len(dataset):,} rows × {n_features} features")
print(f"Label: {label_col} | Entity: {entity_col} | Folds: {len(splits)}")

dataset_pd = dataset.to_pandas()
n_entities = dataset_pd[entity_col].nunique()
print(f"Entities: {n_entities}")

Dataset: 325,089 rows × 51 features
Label: fwd_ret_dh_10d | Entity: symbol | Folds: 2
Entities: 610


## 2. Prior Baselines

Load IC results from earlier pipeline stages (Ch11 linear, Ch12 GBM)
rather than re-running them here.

In [5]:
prior_baselines = {}
_baselines = load_best_ic_per_family(["linear", "gbm"], case_studies=[CASE_STUDY_ID])
if not _baselines.is_empty():
    for row in _baselines.iter_rows(named=True):
        if row["family"] == "linear":
            prior_baselines[f"{row['config_name'].title()} (Ch11)"] = row["ic_mean"]
        elif row["family"] == "gbm":
            prior_baselines["GBM (Ch12)"] = row["ic_mean"]

if prior_baselines:
    for name, ic in prior_baselines.items():
        print(f"  {name}: IC={ic:+.4f}" if ic is not None else f"  {name}: IC=N/A")
else:
    print("  No prior results found — run 06_linear.py and 07_gbm.py first")

  GBM (Ch12): IC=+0.0742
  Ridge_A10000.0 (Ch11): IC=+0.0442


## 3. LSTM

Primary architecture for this notebook.

In [6]:
dl_configs = load_configs(CASE_STUDY_ID, PRIMARY_LABEL, "deep_learning")
dl_configs = [c for c in dl_configs if c["params"].get("architecture") == MODEL]

# Apply Papermill overrides to configs (test mode: fewer epochs)
for cfg in dl_configs:
    if cfg.get("n_epochs", 100) != N_EPOCHS:
        cfg["n_epochs"] = N_EPOCHS
    if cfg.get("batch_size", 2048) != BATCH_SIZE:
        cfg["batch_size"] = BATCH_SIZE
    if cfg["params"].get("lookback", 60) != LOOKBACK:
        cfg["params"]["lookback"] = LOOKBACK

print(
    f"Grid: {len(dl_configs)} configs × {dl_configs[0].get('n_epochs', 100)} epochs × {len(splits)} folds"
)
for cfg in dl_configs:
    print(
        f"  {cfg['config_name']}: {cfg['params'].get('architecture', '?')} ({cfg.get('n_epochs', 100)} epochs)"
    )

Grid: 1 configs × 100 epochs × 2 folds
  lstm_h64: lstm (100 epochs)


In [7]:
result = run_dl_cv(
    dataset_pd,
    splits,
    feature_names=feature_names,
    label_col=label_col,
    date_col=date_col,
    entity_col=entity_col,
    configs=dl_configs,
    n_features=n_features,
    device=device_str,
    save_dir=CASE_DIR / "run_log" / "training" / "deep_learning",
    register=True,
    force_retrain=FORCE_RETRAIN,
    case_study=CASE_STUDY_ID,
    notebook=f"dl_{MODEL}",
    temporal_by_fold=mds.temporal_by_fold,
    temporal_keys=mds.temporal_keys,
    temporal_feature_names=mds.temporal_feature_names,
)

Fold-major CV: 2 folds × 1 configs × 60 lookback

  Fold 0: creating sequences...


    train=94,736 seq across 455 symbols
    val=44,253 seq across 410 symbols
    creating datasets...
    datasets ready
    lstm_h64:


      epoch   1/100: train_loss=0.046641


      epoch   2/100: train_loss=0.044920


      epoch   3/100: train_loss=0.042684


...

      epoch  98/100: train_loss=0.011871


      epoch  99/100: train_loss=0.011831


      epoch 100/100: train_loss=0.011911, val_loss=0.064600, IC=+0.0054


      best_ep=15, IC=+0.0180 (598.9s, 20 checkpoints)


  lstm_h64: best_epoch=5, IC=+0.0456 (987.0s)

  Best: lstm_h64 @ epoch 5 (IC=+0.0456)
  Saved to case_studies/sp500_options/run_log/training/deep_learning


## 4. Learning Curves

In [8]:
grid_results = result["grid_results"]
best_name = result["best_config_name"]
best_epoch = result["best_epoch"]
best_ic = result["best_ic"]

curves = result["all_learning_curves"]
if curves.height > 0:
    checkpoints = sorted(curves["epoch"].unique().to_list())
    display_cps = [cp for cp in checkpoints if cp % 20 == 0 or cp == checkpoints[-1]]

    print(f"{'Config':15s}", end="")
    for cp in display_cps:
        print(f" {cp:>7d}", end="")
    print()
    print("-" * (15 + 8 * len(display_cps)))

    for r in grid_results:
        cfg_data = curves.filter(pl.col("config") == r["config_name"])
        print(f"{r['config_name']:15s}", end="")
        for cp in display_cps:
            row = cfg_data.filter(pl.col("epoch") == cp)
            if row.height > 0:
                print(f" {row['ic_mean'][0]:+7.4f}", end="")
            else:
                print(f" {'N/A':>7s}", end="")
        print()

Config               20      40      60      80     100
-------------------------------------------------------
lstm_h64        +0.0277 +0.0170 +0.0166 +0.0170 +0.0166


## 5. MC Dropout Uncertainty (Optional)

In [9]:
if MC_DROPOUT:
    from ml4t.diagnostic.metrics import cross_sectional_ic

    from case_studies.utils.deep_learning import mc_dropout_predict
    from case_studies.utils.sequence_dataset import (
        materialize_sequences,
        prepare_fold_sequence_stores,
    )

    dates_series = dataset_pd[date_col]
    last_fold = splits[-1]
    train_mask = (dates_series >= last_fold["train_start"]) & (
        dates_series <= last_fold["train_end"]
    )
    val_mask = (dates_series >= last_fold["val_start"]) & (dates_series <= last_fold["val_end"])

    train_store, val_store, _ = prepare_fold_sequence_stores(
        dataset_pd,
        train_mask=train_mask,
        val_mask=val_mask,
        feature_names=feature_names,
        label_col=label_col,
        date_col=date_col,
        entity_col=entity_col,
        lookback=LOOKBACK,
    )
    X_train_seq, y_train_seq, _, _ = materialize_sequences(train_store)
    X_val_seq, y_val_seq, val_dates, val_entities = materialize_sequences(val_store)

    if len(X_train_seq) > 0 and len(X_val_seq) > 0:
        torch_device = torch.device(device_str)
        best_cfg_dict = dl_configs[0]
        arch_name = best_cfg_dict["params"].get(
            "architecture", resolve_arch_name(best_cfg_dict["config_name"])
        )
        from case_studies.utils.deep_learning import build_arch_kwargs

        best_cfg = build_arch_kwargs(
            best_cfg_dict, n_features, best_cfg_dict["params"].get("lookback", 60)
        )
        mc_model = create_model(arch_name, best_cfg).to(torch_device)

        X_t = torch.FloatTensor(X_train_seq).to(torch_device)
        y_t = torch.FloatTensor(y_train_seq).to(torch_device)
        optimizer = torch.optim.AdamW(mc_model.parameters(), lr=1e-3)
        criterion = torch.nn.MSELoss()

        mc_model.train()
        for ep in range(min(N_EPOCHS, 50)):
            idx = torch.randperm(len(X_t))
            for s in range(0, len(X_t), BATCH_SIZE):
                batch = idx[s : s + BATCH_SIZE]
                loss = criterion(mc_model(X_t[batch]), y_t[batch])
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        X_v = torch.FloatTensor(X_val_seq).to(torch_device)
        mean_pred, std_pred = mc_dropout_predict(mc_model, X_v, n_samples=50)

        median_unc = np.median(std_pred)
        low_unc = std_pred <= median_unc
        high_unc = std_pred > median_unc

        low_frame = pl.DataFrame(
            {
                "date": val_dates[low_unc],
                "symbol": val_entities[low_unc],
                "y_true": y_val_seq[low_unc],
                "y_pred": mean_pred[low_unc],
            }
        )
        ic_low = cross_sectional_ic(
            low_frame,
            low_frame,
            pred_col="y_pred",
            ret_col="y_true",
            date_col="date",
            entity_col="symbol",
            min_obs=5,
        )["ic_mean"]
        high_frame = pl.DataFrame(
            {
                "date": val_dates[high_unc],
                "symbol": val_entities[high_unc],
                "y_true": y_val_seq[high_unc],
                "y_pred": mean_pred[high_unc],
            }
        )
        ic_high = cross_sectional_ic(
            high_frame,
            high_frame,
            pred_col="y_pred",
            ret_col="y_true",
            date_col="date",
            entity_col="symbol",
            min_obs=5,
        )["ic_mean"]
        print("MC Dropout uncertainty analysis:")
        print(f"  Low uncertainty IC:  {ic_low:+.4f} ({low_unc.sum():,} samples)")
        print(f"  High uncertainty IC: {ic_high:+.4f} ({high_unc.sum():,} samples)")
        print(f"  IC gap: {ic_low - ic_high:+.4f}")

        del mc_model, X_t, y_t, X_v
        torch.cuda.empty_cache()
else:
    print("MC Dropout disabled (set MC_DROPOUT=True to enable)")

MC Dropout disabled (set MC_DROPOUT=True to enable)


## 6. Comparison

In [10]:
rows = [(name, ic) for name, ic in prior_baselines.items()]
rows.append((best_name, best_ic))

comparison = pl.DataFrame({"Model": [r[0] for r in rows], "IC": [r[1] for r in rows]})
comparison = comparison.with_columns(
    pl.when(pl.col("IC") == pl.col("IC").max())
    .then(pl.lit("*"))
    .otherwise(pl.lit(""))
    .alias("Best")
)
comparison

Model,IC,Best
str,f64,str
"""GBM (Ch12)""",0.074176,"""*"""
"""Ridge_A10000.0 (Ch11)""",0.044249,""""""
"""lstm_h64""",0.045565,""""""


In [11]:
ridge_ic = next((v for k, v in prior_baselines.items() if "ridge" in k.lower()), float("nan"))
dl_delta = best_ic - ridge_ic
print(f"DL delta over Ridge: {dl_delta:+.4f}")

DL delta over Ridge: +nan


## 7. Save Results

Predictions and fold metrics are registered by `run_dl_cv()`
during training. Here we record the pipeline results JSON.

In [12]:
predictions = result["predictions"]
all_predictions = result["all_predictions"]
fold_metrics = result["fold_metrics"]

print(f"Predictions: {predictions.height:,} rows")
print(f"All predictions: {all_predictions.height:,} rows")

Predictions: 78,133 rows
All predictions: 1,562,660 rows


In [13]:
val_ic_mean = float(fold_metrics["ic_mean"].mean()) if fold_metrics.height > 0 else None

## 8. Key Takeaways

1. **LSTM trails ridge and GBM** — gating captures some sequential volatility
   dynamics but does not overcome the flat-feature models on this cross-sectional
   signal. LSTM (IC ~0.037) falls below ridge (~0.044) and well below GBM (~0.068).
2. **PatchTST leads DL**: PatchTST (~0.047) outperforms LSTM (~0.037).
   Patch-based attention captures IV surface dynamics better than gating on
   this cross-sectional data.
3. **Convergence at epoch 10** with 612 symbols confirms that options IV
   features provide an exceptionally dense signal — DL models saturate fast.
4. **Architecture ranking** for options: GBM (0.068) >> PatchTST (0.047) >
   TabM (0.045) ~ ridge (0.044) > LSTM (0.037). The cross-sectional signal
   strongly favors tabular models over sequence architectures.